# OmniOC Lakehouse — Sample Data Generator

Generates all batch files needed for the lakehouse project and uploads them to ADLS.

**Files generated:**
- `landing/supply_chain/` — suppliers.csv, products.csv, warehouses.csv, shipments.parquet, supplier_contracts.json
- `landing/finance/` — customers.json, orders.json, order_lines.json, payments.json, refunds.json, exchange_rates.csv
- `landing/cdc/` — cdc_customers_changes.json, cdc_products_changes.json
- `landing/bad_data/` — poison_pills_orders.json

**Run cells in order top to bottom.**

## Cell 1 — Install Dependencies

In [1]:
!pip install faker pandas pyarrow azure-storage-blob --quiet
print('✅ Dependencies installed')

✅ Dependencies installed


## Cell 2 — Configuration
Edit `OUTPUT_DIR` and `STORAGE_CONNECTION_STRING` before running.

In [2]:
import os

# ── Output directory (local) ──────────────────────────────────────────────────
OUTPUT_DIR = r'C:\omnioc\sample_data'   # change if needed

# ── ADLS Upload config ────────────────────────────────────────────────────────
UPLOAD_TO_ADLS = True    # set False to skip upload and only generate files locally

STORAGE_CONNECTION_STRING = (
    "DefaultEndpointsProtocol=https;"
    "AccountName=omniocdeveusst01;"
    "AccountKey=Bi8diC3w7NK0S85oOdsV0N/+adlVwsMmwMtvNATUWySuFg296pUfeG7NdDLjSLSnaxmaYWJeCu/m+AStA5F/zQ==;"
    "EndpointSuffix=core.windows.net"
)
ADLS_CONTAINER = 'landing'

# ── Data volume controls ──────────────────────────────────────────────────────
N_SUPPLIERS   = 500
N_PRODUCTS    = 5000
N_WAREHOUSES  = 20
N_SHIPMENTS   = 30000
N_CUSTOMERS   = 10000
N_ORDERS      = 100000
N_CONTRACTS   = 400
N_REFUNDS     = 2000
N_CDC_CUST    = 2000
N_CDC_PROD    = 500
N_POISON      = 500

print('✅ Config ready')
print(f'   Output dir     : {OUTPUT_DIR}')
print(f'   Upload to ADLS : {UPLOAD_TO_ADLS}')

✅ Config ready
   Output dir     : C:\omnioc\sample_data
   Upload to ADLS : True


## Cell 3 — Shared Setup (imports, helpers, reference data)

In [3]:
import json
import random
import uuid
from datetime import datetime, timedelta, timezone
import pandas as pd
from faker import Faker

random.seed(42)
fake = Faker('en_US')
Faker.seed(42)
NOW = datetime.now(timezone.utc)

# Reference data
CATEGORIES       = ['Electronics','Clothing','Home & Garden','Sports','Books',
                    'Food & Beverage','Automotive','Beauty','Toys','Office']
SUBCATEGORIES    = {
    'Electronics':    ['Phones','Laptops','Tablets','Accessories','Cameras'],
    'Clothing':       ['Men','Women','Kids','Shoes','Accessories'],
    'Home & Garden':  ['Furniture','Kitchen','Decor','Garden','Lighting'],
    'Sports':         ['Fitness','Outdoor','Team Sports','Water Sports','Cycling'],
    'Books':          ['Fiction','Non-Fiction','Technical','Children','Academic'],
    'Food & Beverage':['Snacks','Beverages','Organic','Frozen','Supplements'],
    'Automotive':     ['Parts','Accessories','Tools','Electronics','Care'],
    'Beauty':         ['Skincare','Haircare','Makeup','Fragrance','Personal Care'],
    'Toys':           ['Action Figures','Board Games','Educational','Outdoor','Puzzles'],
    'Office':         ['Stationery','Furniture','Tech','Storage','Supplies'],
}
COUNTRIES        = ['US','UK','DE','FR','CA','AU','JP','SG','IN','BR']
CURRENCIES       = ['USD','EUR','GBP','AUD','CAD']
PAYMENT_METHODS  = ['credit_card','debit_card','paypal','bank_transfer','crypto']
ORDER_STATUSES   = ['placed','confirmed','processing','shipped','delivered','cancelled','returned']
CUSTOMER_SEGS    = ['B2C_Regular','B2C_Premium','B2B_SMB','B2B_Enterprise']
REGIONS          = ['Northeast','Southeast','Midwest','West','Southwest','Northwest']
WAREHOUSE_TYPES  = ['Distribution Center','Fulfillment Center','Regional Hub','Last Mile']

def rand_ts(days_back=730, days_forward=0):
    delta = timedelta(days=random.randint(-days_back, -days_forward),
                      hours=random.randint(0,23), minutes=random.randint(0,59))
    return (NOW + delta).isoformat()

def rand_date(days_back=730, days_forward=0):
    delta = timedelta(days=random.randint(-days_back, -days_forward))
    return (NOW + delta).strftime('%Y-%m-%d')

def save_json(data, path, lines=False):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        if lines:
            for row in data:
                f.write(json.dumps(row, default=str) + '\n')
        else:
            json.dump(data, f, indent=2, default=str)
    print(f'  ✅ {os.path.basename(path):45s} {len(data):>7,} records')

def save_csv(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    print(f'  ✅ {os.path.basename(path):45s} {len(df):>7,} rows')

def save_parquet(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)
    print(f'  ✅ {os.path.basename(path):45s} {len(df):>7,} rows')

print('✅ Shared setup complete')

✅ Shared setup complete


## Cell 4 — Generate Supply Chain Data

In [4]:
print('📦 Generating Supply Chain data...')

# ── Suppliers ─────────────────────────────────────────────
suppliers_rows = []
for i in range(1, N_SUPPLIERS + 1):
    suppliers_rows.append({
        'supplier_id': f'SUP-{i:05d}',
        'supplier_name': fake.company(),
        'country': random.choice(COUNTRIES),
        'city': fake.city(),
        'contact_email': fake.company_email(),
        'contact_phone': fake.phone_number(),
        'rating': round(random.uniform(2.0, 5.0), 1),
        'lead_time_days': random.randint(1, 45),
        'payment_terms_days': random.choice([7,14,30,45,60,90]),
        'is_active': random.choices([True, False], weights=[90,10])[0],
        'contract_start_date': rand_date(1800, 365),
        'created_at': rand_ts(1800, 730),
        'updated_at': rand_ts(365, 0),
    })
suppliers_df  = pd.DataFrame(suppliers_rows)
supplier_ids  = suppliers_df['supplier_id'].tolist()

# ── Products ──────────────────────────────────────────────
products_rows = []
for i in range(1, N_PRODUCTS + 1):
    cat   = random.choice(CATEGORIES)
    subcat = random.choice(SUBCATEGORIES[cat])
    cost  = round(random.uniform(1.5, 500.0), 2)
    products_rows.append({
        'product_id': f'PRD-{i:06d}',
        'product_name': f"{fake.word().capitalize()} {subcat} {fake.word().capitalize()} {random.randint(1,999)}",
        'sku': f"{cat[:3].upper()}-{subcat[:3].upper()}-{random.randint(10000,99999)}",
        'category': cat,
        'sub_category': subcat,
        'unit_cost': cost,
        'unit_price': round(cost * random.uniform(1.2, 3.5), 2),
        'weight_kg': round(random.uniform(0.05, 25.0), 3),
        'is_active': random.choices([True, False], weights=[88,12])[0],
        'supplier_id': random.choice(supplier_ids),
        'reorder_point': random.randint(5, 100),
        'reorder_quantity': random.randint(50, 500),
        'created_at': rand_ts(1800, 365),
        'updated_at': rand_ts(365, 0),
    })
products_df  = pd.DataFrame(products_rows)
product_ids  = products_df['product_id'].tolist()

# ── Warehouses ────────────────────────────────────────────
warehouses_rows = []
for i in range(1, N_WAREHOUSES + 1):
    warehouses_rows.append({
        'warehouse_id': f'WH-{i:03d}',
        'warehouse_name': f"{fake.city()} {random.choice(WAREHOUSE_TYPES)}",
        'region': random.choice(REGIONS),
        'country': 'US',
        'city': fake.city(),
        'state': fake.state_abbr(),
        'zip_code': fake.zipcode(),
        'address': fake.street_address(),
        'capacity_sqft': random.randint(10000, 500000),
        'max_sku_capacity': random.randint(500, 50000),
        'manager_name': fake.name(),
        'manager_email': fake.email(),
        'is_active': True,
        'opened_date': rand_date(3650, 365),
    })
warehouses_df = pd.DataFrame(warehouses_rows)
warehouse_ids = warehouses_df['warehouse_id'].tolist()

# ── Shipments ─────────────────────────────────────────────
shipments_rows = []
for i in range(1, N_SHIPMENTS + 1):
    ship_date = (NOW + timedelta(days=random.randint(-400,-1))).strftime('%Y-%m-%d')
    eta       = (datetime.strptime(ship_date,'%Y-%m-%d') + timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d')
    arrival   = (datetime.strptime(eta,'%Y-%m-%d') + timedelta(days=random.randint(-2,5))).strftime('%Y-%m-%d')
    shipments_rows.append({
        'shipment_id': f'SHIP-{i:07d}',
        'supplier_id': random.choice(supplier_ids),
        'destination_warehouse_id': random.choice(warehouse_ids),
        'product_id': random.choice(product_ids),
        'quantity': random.randint(10, 2000),
        'unit_cost': round(random.uniform(1.5, 500.0), 2),
        'total_cost': round(random.uniform(100, 50000), 2),
        'ship_date': ship_date,
        'eta_date': eta,
        'actual_arrival_date': arrival,
        'carrier': random.choice(['FedEx','UPS','DHL','USPS','Amazon Logistics']),
        'tracking_number': fake.uuid4()[:16].upper(),
        'status': random.choices(['delivered','in_transit','delayed','lost'], weights=[80,10,8,2])[0],
        'created_at': rand_ts(400, 1),
    })
shipments_df = pd.DataFrame(shipments_rows)

# ── Supplier Contracts ────────────────────────────────────
contracts = []
for i in range(1, N_CONTRACTS + 1):
    start = rand_date(1800, 365)
    end   = (datetime.strptime(start,'%Y-%m-%d') + timedelta(days=random.randint(365,1095))).strftime('%Y-%m-%d')
    contracts.append({
        'contract_id': f'CTR-{i:05d}',
        'supplier_id': random.choice(supplier_ids),
        'contract_type': random.choice(['Exclusive','Non-Exclusive','Preferred','Spot']),
        'start_date': start,
        'end_date': end,
        'total_value_usd': round(random.uniform(50000, 5000000), 2),
        'payment_currency': random.choice(CURRENCIES),
        'sla_terms': {
            'lead_time_days': random.randint(1,30),
            'defect_rate_max_pct': round(random.uniform(0.1,2.0),2),
            'on_time_delivery_target_pct': random.choice([95,97,98,99]),
            'penalty_per_day_late_usd': random.randint(100,5000),
        },
        'auto_renewal': random.choice([True,False]),
        'is_active': random.choices([True,False], weights=[85,15])[0],
        'created_at': rand_ts(1800,365),
    })

# ── Save supply chain files ───────────────────────────────
save_csv(suppliers_df,  f'{OUTPUT_DIR}/landing/supply_chain/suppliers/suppliers.csv')
save_csv(products_df,   f'{OUTPUT_DIR}/landing/supply_chain/products/products.csv')
save_csv(warehouses_df, f'{OUTPUT_DIR}/landing/supply_chain/warehouses/warehouses.csv')
save_parquet(shipments_df, f'{OUTPUT_DIR}/landing/supply_chain/shipments/shipments.parquet')
save_json(contracts, f'{OUTPUT_DIR}/landing/supply_chain/supplier_contracts/supplier_contracts.json')

print('\n✅ Supply chain data complete')

📦 Generating Supply Chain data...
  ✅ suppliers.csv                                     500 rows
  ✅ products.csv                                    5,000 rows
  ✅ warehouses.csv                                     20 rows
  ✅ shipments.parquet                              30,000 rows
  ✅ supplier_contracts.json                           400 records

✅ Supply chain data complete


## Cell 5 — Generate Finance / Transaction Data

In [5]:
print('💰 Generating Finance / Transaction data...')

# ── Customers ─────────────────────────────────────────────
customers = []
for i in range(1, N_CUSTOMERS + 1):
    customers.append({
        'customer_id': f'CUST-{i:07d}',
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'email': fake.email(),
        'phone': fake.phone_number(),
        'date_of_birth': fake.date_of_birth(minimum_age=18, maximum_age=80).isoformat(),
        'gender': random.choice(['M','F','Other',None]),
        'segment': random.choices(CUSTOMER_SEGS, weights=[50,20,20,10])[0],
        'country': random.choice(COUNTRIES),
        'city': fake.city(),
        'state': fake.state_abbr(),
        'zip_code': fake.zipcode(),
        'address_line1': fake.street_address(),
        'signup_date': rand_date(1800, 30),
        'loyalty_points': random.randint(0, 50000),
        'preferred_currency': random.choice(CURRENCIES),
        'is_email_verified': random.choices([True,False], weights=[85,15])[0],
        'is_active': random.choices([True,False], weights=[90,10])[0],
        'created_at': rand_ts(1800, 30),
        'updated_at': rand_ts(30, 0),
    })
customer_ids = [c['customer_id'] for c in customers]
save_json(customers, f'{OUTPUT_DIR}/landing/finance/customers/customers.json', lines=True)

# ── Orders ────────────────────────────────────────────────
orders = []
for i in range(1, N_ORDERS + 1):
    order_date = rand_ts(730, 0)
    status     = random.choices(ORDER_STATUSES, weights=[5,10,10,20,45,7,3])[0]
    orders.append({
        'order_id': f'ORD-{i:08d}',
        'customer_id': random.choice(customer_ids),
        'warehouse_id': random.choice(warehouse_ids),
        'order_date': order_date,
        'status': status,
        'channel': random.choice(['web','mobile_app','marketplace','in_store','api']),
        'total_amount': round(random.uniform(5, 5000), 2),
        'discount_amount': round(random.uniform(0, 200), 2),
        'tax_amount': round(random.uniform(0, 400), 2),
        'shipping_amount': round(random.uniform(0, 50), 2),
        'currency': random.choice(CURRENCIES),
        'shipping_address_city': fake.city(),
        'shipping_address_country': random.choice(COUNTRIES),
        'ship_date': rand_ts(720,0) if status in ['shipped','delivered'] else None,
        'delivery_date': rand_ts(700,0) if status == 'delivered' else None,
        'promo_code': f'PROMO{random.randint(100,999)}' if random.random() < 0.15 else None,
        'created_at': order_date,
        'updated_at': rand_ts(30, 0),
    })
save_json(orders, f'{OUTPUT_DIR}/landing/finance/orders/orders.json', lines=True)

# ── Order Lines ───────────────────────────────────────────
order_lines = []
line_id = 1
for order in orders[:50000]:   # subset to keep memory manageable
    n_lines = random.choices([1,2,3,4,5], weights=[40,30,15,10,5])[0]
    for _ in range(n_lines):
        qty        = random.randint(1, 10)
        unit_price = round(random.uniform(2, 800), 2)
        order_lines.append({
            'line_id': f'LINE-{line_id:09d}',
            'order_id': order['order_id'],
            'product_id': random.choice(product_ids),
            'quantity': qty,
            'unit_price': unit_price,
            'discount_pct': random.choices([0,5,10,15,20], weights=[60,15,12,8,5])[0],
            'line_total': round(qty * unit_price, 2),
            'is_gift': random.choices([False,True], weights=[92,8])[0],
            'gift_message': fake.sentence() if random.random() < 0.05 else None,
            'created_at': order['created_at'],
            'updated_at': rand_ts(30, 0),
        })
        line_id += 1
save_json(order_lines, f'{OUTPUT_DIR}/landing/finance/order_lines/order_lines.json', lines=True)

# ── Payments ──────────────────────────────────────────────
payments = []
for order in orders[:80000]:
    if order['status'] not in ['placed']:
        method = random.choice(PAYMENT_METHODS)
        payments.append({
            'payment_id': f"PAY-{fake.uuid4()[:12].upper()}",
            'order_id': order['order_id'],
            'amount': order['total_amount'],
            'currency': order['currency'],
            'method': method,
            'status': random.choices(['completed','pending','failed','refunded'], weights=[80,10,5,5])[0],
            'gateway': random.choice(['Stripe','PayPal','Braintree','Square','Adyen']),
            'transaction_id': f"TXN-{fake.uuid4()[:16].upper()}",
            'card_last_four': str(random.randint(1000,9999)) if 'card' in method else None,
            'billing_country': random.choice(COUNTRIES),
            'created_at': order['created_at'],
            'paid_at': rand_ts(700, 0),
            'updated_at': rand_ts(30, 0),
        })
save_json(payments, f'{OUTPUT_DIR}/landing/finance/payments/payments.json', lines=True)

# ── Refunds ───────────────────────────────────────────────
eligible = [o for o in orders if o['status'] in ['delivered','returned']]
refund_sample = random.sample(eligible, min(N_REFUNDS, len(eligible)))
refunds = []
for order in refund_sample:
    refunds.append({
        'refund_id': f"REF-{fake.uuid4()[:12].upper()}",
        'order_id': order['order_id'],
        'refund_amount': round(random.uniform(5, order['total_amount']), 2),
        'currency': order['currency'],
        'reason': random.choice(['damaged_product','wrong_item','not_as_described',
                                  'changed_mind','late_delivery','duplicate_order']),
        'refund_type': random.choice(['full','partial']),
        'status': random.choices(['approved','pending','rejected'], weights=[75,20,5])[0],
        'requested_at': rand_ts(60, 0),
        'processed_at': rand_ts(50, 0),
        'notes': fake.sentence() if random.random() < 0.3 else None,
    })
save_json(refunds, f'{OUTPUT_DIR}/landing/finance/refunds/refunds.json', lines=True)

# ── Exchange Rates ────────────────────────────────────────
base_rates = {'EUR':0.92,'GBP':0.79,'AUD':1.53,'CAD':1.36,'JPY':149.5,'SGD':1.34}
fx_rows = []
for i in range(365):
    date = (NOW - timedelta(days=i)).strftime('%Y-%m-%d')
    for cur, rate in base_rates.items():
        fx_rows.append({
            'rate_date': date,
            'base_currency': 'USD',
            'target_currency': cur,
            'exchange_rate': round(rate * random.uniform(0.97,1.03), 6),
            'source': 'ECB',
            'created_at': date + 'T00:00:00Z',
        })
fx_df = pd.DataFrame(fx_rows)
save_csv(fx_df, f'{OUTPUT_DIR}/landing/finance/exchange_rates/exchange_rates.csv')

print('\n✅ Finance data complete')

💰 Generating Finance / Transaction data...
  ✅ customers.json                                 10,000 records
  ✅ orders.json                                   100,000 records
  ✅ order_lines.json                              104,908 records
  ✅ payments.json                                  76,132 records
  ✅ refunds.json                                    2,000 records
  ✅ exchange_rates.csv                              2,190 rows

✅ Finance data complete


## Cell 6 — Generate CDC Change Records

In [6]:
print('🔄 Generating CDC change records...')

# ── CDC Customers ─────────────────────────────────────────
cdc_customers = []
for _ in range(N_CDC_CUST):
    cid       = random.choice(customer_ids)
    operation = random.choices(['INSERT','UPDATE','DELETE'], weights=[20,75,5])[0]
    ts        = rand_ts(30, 0)
    before    = None if operation == 'INSERT' else {
        'customer_id': cid, 'email': fake.email(),
        'loyalty_points': random.randint(0,50000),
        'segment': random.choice(CUSTOMER_SEGS),
        'is_active': True, 'updated_at': rand_ts(60,30)
    }
    after = None if operation == 'DELETE' else {
        'customer_id': cid, 'email': fake.email(),
        'loyalty_points': random.randint(0,50000),
        'segment': random.choice(CUSTOMER_SEGS),
        'is_active': operation != 'DELETE', 'updated_at': ts
    }
    cdc_customers.append({
        'cdc_event_id': str(uuid.uuid4()),
        'operation': operation,
        'table_name': 'dbo.customers',
        'commit_timestamp': ts,
        'lsn': f"{random.randint(0,99999999):010d}:{random.randint(0,9999):05d}:{random.randint(1,5):04d}",
        'before': before,
        'after': after,
    })
save_json(cdc_customers, f'{OUTPUT_DIR}/landing/cdc/customers/cdc_customers_changes.json', lines=True)

# ── CDC Products ──────────────────────────────────────────
cdc_products = []
for _ in range(N_CDC_PROD):
    pid       = random.choice(product_ids)
    operation = random.choices(['UPDATE','INSERT'], weights=[85,15])[0]
    ts        = rand_ts(30, 0)
    old_price = round(random.uniform(5,800), 2)
    cdc_products.append({
        'cdc_event_id': str(uuid.uuid4()),
        'operation': operation,
        'table_name': 'dbo.products',
        'commit_timestamp': ts,
        'lsn': f"{random.randint(0,99999999):010d}:{random.randint(0,9999):05d}:{random.randint(1,5):04d}",
        'before': None if operation == 'INSERT' else {
            'product_id': pid, 'unit_price': old_price,
            'is_active': True, 'updated_at': rand_ts(60,30)
        },
        'after': {
            'product_id': pid,
            'unit_price': round(old_price * random.uniform(0.8,1.3), 2),
            'is_active': random.choices([True,False], weights=[90,10])[0],
            'updated_at': ts,
        },
    })
save_json(cdc_products, f'{OUTPUT_DIR}/landing/cdc/products/cdc_products_changes.json', lines=True)

print('\n✅ CDC data complete')

🔄 Generating CDC change records...
  ✅ cdc_customers_changes.json                      2,000 records
  ✅ cdc_products_changes.json                         500 records

✅ CDC data complete


## Cell 7 — Generate Poison Pills (Bad Data)

In [7]:
print('⚠️  Generating poison pill records (intentionally bad data for DQ testing)...')

poison_pills = []
for i in range(N_POISON):
    pill_type = random.choice(['null_required','wrong_type','out_of_range','schema_drift','duplicate'])
    if pill_type == 'null_required':
        poison_pills.append({
            'order_id': None,
            'customer_id': f'CUST-{random.randint(1,10000):07d}',
            'total_amount': 99.99,
            'order_date': rand_ts(),
            '_dq_issue': 'null_order_id'
        })
    elif pill_type == 'wrong_type':
        poison_pills.append({
            'order_id': f'ORD-BAD-{i}',
            'customer_id': 'NOT_A_VALID_ID',
            'total_amount': 'NOT_A_NUMBER',
            'order_date': rand_ts(),
            '_dq_issue': 'type_mismatch_amount'
        })
    elif pill_type == 'out_of_range':
        poison_pills.append({
            'order_id': f'ORD-RANGE-{i}',
            'customer_id': f'CUST-{random.randint(1,10000):07d}',
            'total_amount': -999.99,
            'order_date': rand_ts(),
            '_dq_issue': 'negative_amount'
        })
    elif pill_type == 'schema_drift':
        poison_pills.append({
            'order_id': f'ORD-NEW-{i}',
            'customer_id': f'CUST-{random.randint(1,10000):07d}',
            'total_amount': 45.00,
            'order_date': rand_ts(),
            'new_field_not_in_schema': 'unexpected_value',
            'another_new_field': 12345,
            '_dq_issue': 'schema_drift'
        })
    elif pill_type == 'duplicate':
        poison_pills.append({
            'order_id': f'ORD-{random.randint(1,1000):08d}',
            'customer_id': f'CUST-{random.randint(1,10000):07d}',
            'total_amount': 45.00,
            'order_date': rand_ts(),
            '_dq_issue': 'duplicate_order_id'
        })

save_json(poison_pills, f'{OUTPUT_DIR}/landing/bad_data/poison_pills_orders.json', lines=True)
print('\n✅ Poison pills complete')

⚠️  Generating poison pill records (intentionally bad data for DQ testing)...
  ✅ poison_pills_orders.json                          500 records

✅ Poison pills complete


## Cell 8 — Summary of Generated Files

In [8]:
print('\n' + '='*65)
print('  📊 DATA GENERATION COMPLETE — File Summary')
print('='*65)

all_files = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for fname in files:
        fpath = os.path.join(root, fname)
        size_kb = os.path.getsize(fpath) / 1024
        rel_path = os.path.relpath(fpath, OUTPUT_DIR)
        all_files.append((rel_path, size_kb))
        print(f'  ✅ {rel_path:<65} {size_kb:>8.1f} KB')

total_mb = sum(s for _, s in all_files) / 1024
print('='*65)
print(f'  Total files : {len(all_files)}')
print(f'  Total size  : {total_mb:.1f} MB')
print(f'  Output dir  : {os.path.abspath(OUTPUT_DIR)}')
print('='*65)
print('\n👉 Next: Run Cell 9 to upload all files to ADLS')


  📊 DATA GENERATION COMPLETE — File Summary
  ✅ landing\bad_data\poison_pills_orders.json                             88.2 KB
  ✅ landing\cdc\customers\cdc_customers_changes.json                    1072.0 KB
  ✅ landing\cdc\products\cdc_products_changes.json                       215.5 KB
  ✅ landing\finance\customers\customers.json                            5531.8 KB
  ✅ landing\finance\exchange_rates\exchange_rates.csv                    116.0 KB
  ✅ landing\finance\orders\orders.json                                 55652.3 KB
  ✅ landing\finance\order_lines\order_lines.json                       31462.5 KB
  ✅ landing\finance\payments\payments.json                             30310.1 KB
  ✅ landing\finance\refunds\refunds.json                                 605.5 KB
  ✅ landing\supply_chain\products\products.csv                           871.2 KB
  ✅ landing\supply_chain\shipments\shipments.parquet                    1702.7 KB
  ✅ landing\supply_chain\suppliers\suppliers.csv     

## Cell 9 — Upload to ADLS
Make sure `STORAGE_CONNECTION_STRING` is set correctly in Cell 2 before running.

In [9]:
if not UPLOAD_TO_ADLS:
    print('⏭️  UPLOAD_TO_ADLS = False — skipping upload')
else:
    from azure.storage.blob import BlobServiceClient

    print('☁️  Uploading files to ADLS...')
    print(f'   Container: {ADLS_CONTAINER}')
    print(f'   Storage  : omniocdeveusst01\n')

    client           = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
    container_client = client.get_container_client(ADLS_CONTAINER)

    uploaded = []
    failed   = []

    for root, dirs, files in os.walk(OUTPUT_DIR):
        for fname in files:
            local_path = os.path.join(root, fname)
            # Build blob path: strip OUTPUT_DIR prefix, keep landing/... structure
            rel_path   = os.path.relpath(local_path, OUTPUT_DIR)
            blob_path  = rel_path.replace('\\', '/')   # Windows path fix

            try:
                with open(local_path, 'rb') as f:
                    container_client.upload_blob(
                        name      = blob_path,
                        data      = f,
                        overwrite = True
                    )
                size_kb = os.path.getsize(local_path) / 1024
                print(f'  ✅ {blob_path:<65} {size_kb:>7.1f} KB')
                uploaded.append(blob_path)
            except Exception as e:
                print(f'  ❌ FAILED: {blob_path} — {e}')
                failed.append(blob_path)

    print('\n' + '='*65)
    print(f'  ✅ Uploaded : {len(uploaded)} files')
    print(f'  ❌ Failed   : {len(failed)} files')
    if failed:
        print(f'  Failed files: {failed}')
    print('='*65)
    print('\n✅ All files in ADLS landing/ container')
    print('   Databricks Autoloader will pick them up automatically')

☁️  Uploading files to ADLS...
   Container: landing
   Storage  : omniocdeveusst01

  ✅ landing/bad_data/poison_pills_orders.json                            88.2 KB
  ✅ landing/cdc/customers/cdc_customers_changes.json                   1072.0 KB
  ✅ landing/cdc/products/cdc_products_changes.json                      215.5 KB
  ✅ landing/finance/customers/customers.json                           5531.8 KB
  ✅ landing/finance/exchange_rates/exchange_rates.csv                   116.0 KB
  ✅ landing/finance/orders/orders.json                                55652.3 KB
  ✅ landing/finance/order_lines/order_lines.json                      31462.5 KB
  ✅ landing/finance/payments/payments.json                            30310.1 KB
  ✅ landing/finance/refunds/refunds.json                                605.5 KB
  ✅ landing/supply_chain/products/products.csv                          871.2 KB
  ✅ landing/supply_chain/shipments/shipments.parquet                   1702.7 KB
  ✅ landing/supply_chain

## Cell 10 — Verify Upload in ADLS (Optional)

In [10]:
if UPLOAD_TO_ADLS:
    from azure.storage.blob import BlobServiceClient

    client           = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
    container_client = client.get_container_client(ADLS_CONTAINER)

    print('🔍 Files in ADLS landing/ container:\n')
    blobs = list(container_client.list_blobs())
    for blob in sorted(blobs, key=lambda b: b.name):
        size_kb = blob.size / 1024
        print(f'  {blob.name:<70} {size_kb:>8.1f} KB')

    print(f'\n✅ Total blobs in landing/: {len(blobs)}')
else:
    print('⏭️  Skipped — UPLOAD_TO_ADLS = False')

🔍 Files in ADLS landing/ container:

  landing                                                                     0.0 KB
  landing/bad_data                                                            0.0 KB
  landing/bad_data/poison_pills_orders.json                                  88.2 KB
  landing/cdc                                                                 0.0 KB
  landing/cdc/customers                                                       0.0 KB
  landing/cdc/customers/cdc_customers_changes.json                         1072.0 KB
  landing/cdc/products                                                        0.0 KB
  landing/cdc/products/cdc_products_changes.json                            215.5 KB
  landing/finance                                                             0.0 KB
  landing/finance/customers                                                   0.0 KB
  landing/finance/customers/customers.json                                 5531.8 KB
  landing/finance/exchange_r